# Segmentação Semântica

Um detector diz o que há na imagem e onde, com uma caixa em volta de cada objeto. Na **segmentação semântica**, a rede classifica **cada pixel**, e a saída é uma **máscara** do tamanho da imagem em que cada pixel recebe a classe do objeto a que pertence, ou "fundo". É o que se usa quando a forma importa, como ao delimitar um tumor em um exame ou a pista em uma imagem de carro autônomo.

Vamos treinar do zero uma **U-Net**, a arquitetura mais usada para segmentação, com os mesmos dados sintéticos do material de detecção.

In [ ]:
import random

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.colors import ListedColormap
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## O dataset sintético

Como no detector, cada imagem tem $112 \times 112$ pixels e de 1 a 4 dígitos do MNIST colados sem se sobrepor. A anotação agora é uma máscara $112 \times 112$: o pixel vale 0 no fundo e $d + 1$ quando pertence a um dígito $d$, em 11 classes. Um pixel pertence ao dígito quando a intensidade dele passa de 0,5.

In [ ]:
IMAGE_SIZE = 112
NUM_CLASSES = 11   # fundo e os dígitos de 0 a 9

mnist_train = datasets.MNIST(root="data", train=True, download=True, transform=transforms.ToTensor())
mnist_test = datasets.MNIST(root="data", train=False, download=True, transform=transforms.ToTensor())


def overlaps(a, b):
    ax, ay, a_size = a
    bx, by, b_size = b
    return ax < bx + b_size and bx < ax + a_size and ay < by + b_size and by < ay + a_size


def make_image(mnist, rng, max_digits=4):
    image = torch.zeros(1, IMAGE_SIZE, IMAGE_SIZE)
    mask = torch.zeros(IMAGE_SIZE, IMAGE_SIZE, dtype=torch.long)
    placed = []   # (x0, y0, tamanho) em pixels

    n_digits = rng.randint(1, max_digits)
    for _ in range(100):   # tentativas
        if len(placed) == n_digits:
            break

        size = rng.randint(28, 56)
        x0 = rng.randint(0, IMAGE_SIZE - size)
        y0 = rng.randint(0, IMAGE_SIZE - size)
        if any(overlaps((x0, y0, size), other) for other in placed):
            continue

        digit, label = mnist[rng.randrange(len(mnist))]                             # [1, 28, 28]
        digit = F.interpolate(digit[None], size=(size, size), mode="bilinear")[0]   # [1, size, size]
        image[:, y0:y0 + size, x0:x0 + size] = digit
        mask[y0:y0 + size, x0:x0 + size][digit[0] > 0.5] = label + 1               # pixels do dígito

        placed.append((x0, y0, size))

    return image, mask


class SyntheticMNIST(Dataset):
    def __init__(self, mnist, num_images, seed):
        self.mnist = mnist
        self.num_images = num_images
        self.seed = seed

    def __len__(self):
        return self.num_images

    def __getitem__(self, idx):
        rng = random.Random(self.seed * 1_000_000 + idx)
        return make_image(self.mnist, rng)


train_set = SyntheticMNIST(mnist_train, num_images=10000, seed=0)
test_set = SyntheticMNIST(mnist_test, num_images=1000, seed=1)

train_dataloader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2)
test_dataloader = DataLoader(test_set, batch_size=100, num_workers=2)

images, masks = next(iter(train_dataloader))
print(images.shape)   # [64, 1, 112, 112]
print(masks.shape)    # [64, 112, 112]

Na máscara, o fundo fica preto e cada dígito tem a sua cor.

In [ ]:
MASK_CMAP = ListedColormap(["black", *plt.cm.tab10.colors])


def show_mask(ax, mask):
    ax.imshow(mask, cmap=MASK_CMAP, vmin=0, vmax=NUM_CLASSES - 1, interpolation="nearest")
    ax.axis("off")


fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for k in range(4):
    axes[0, k].imshow(images[k, 0], cmap="gray")
    axes[0, k].set_title("imagem")
    axes[0, k].axis("off")
    show_mask(axes[1, k], masks[k])
    axes[1, k].set_title("máscara")
plt.show()

## Medindo a qualidade

A rede devolve um logit por classe em cada pixel, uma saída $[N, 11, H, W]$, e a perda é a entropia cruzada de sempre, com a média sobre todos os pixels. O `F.cross_entropy` já aceita esse formato, com o alvo $[N, H, W]$.

A medida mais óbvia é a **acurácia de pixels**. Mas quase todos os pixels são fundo. Tente prever: qual a acurácia de um "modelo" que responde fundo em todos os pixels?

In [ ]:
all_background = torch.zeros_like(masks)
print(f"acurácia de pixels: {(all_background == masks).float().mean():.1%}")

A medida padrão é a **IoU** do material de detecção, agora entre conjuntos de pixels. Para cada classe $c$:

$$
\text{IoU}_c = \frac{TP_c}{TP_c + FP_c + FN_c}
$$

As contagens saem da matriz de confusão dos pixels, com a classe real na linha e a prevista na coluna: $TP_c$ é a diagonal, $FP_c$ o resto da coluna e $FN_c$ o resto da linha. A **mIoU** é a média sobre as classes, e cada classe pesa o mesmo, por menor que seja.

In [ ]:
def pixel_confusion(predictions, targets):
    index = targets.flatten() * NUM_CLASSES + predictions.flatten()   # par (real, previsto) vira um índice
    return torch.bincount(index, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)


def iou_per_class(confusion):
    tp = confusion.diag()
    fp = confusion.sum(dim=0) - tp
    fn = confusion.sum(dim=1) - tp
    return tp / (tp + fp + fn)


iou = iou_per_class(pixel_confusion(all_background, masks))
print(f"mIoU: {iou.mean():.3f}")

## U-Net

A rede precisa de uma saída do tamanho da entrada, mas também precisa enxergar o dígito inteiro para saber qual é. A U-Net tem duas metades. O **encoder** é uma CNN comum: convoluções e pooling levam a imagem de $112 \times 112$ até $7 \times 7$. O **decoder** volta à resolução cheia com **convoluções transpostas** (`nn.ConvTranspose2d`), que, com kernel 2 e stride 2, transformam cada pixel em um bloco $2 \times 2$ aprendido e dobram a resolução.

Só que no $7 \times 7$ a posição exata de cada traço já se perdeu. Por isso a U-Net tem **conexões de atalho** (*skip connections*): o mapa de cada nível do encoder é **concatenado** ao mapa do decoder de mesma resolução. O que vem de baixo diz **o que** é cada região, e o atalho diz **onde** estão as bordas.

```
112x112  enc1 ─────────────────────── concat → dec1 → head
 56x56     enc2 ───────────────── concat → dec2
 28x28       enc3 ─────────── concat → dec3
 14x14         enc4 ───── concat → dec4
  7x7            bottleneck
```

Depois do `torch.cat`, o mapa tem o dobro de canais, e é isso que cada bloco do decoder recebe.

In [ ]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
    )


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = conv_block(1, 16)        # [N, 16, 112, 112]
        self.enc2 = conv_block(16, 32)       # [N, 32, 56, 56]
        self.enc3 = conv_block(32, 64)       # [N, 64, 28, 28]
        self.enc4 = conv_block(64, 128)      # [N, 128, 14, 14]
        self.bottleneck = conv_block(128, 256)   # [N, 256, 7, 7]
        self.pool = nn.MaxPool2d(2)

        self.up4 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec4 = conv_block(256, 128)     # 128 do up4 + 128 do enc4
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec3 = conv_block(128, 64)
        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = conv_block(64, 32)
        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec1 = conv_block(32, 16)
        self.head = nn.Conv2d(16, NUM_CLASSES, kernel_size=1)   # [N, 11, 112, 112]

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        x = self.bottleneck(self.pool(e4))

        x = self.dec4(torch.cat([self.up4(x), e4], dim=1))   # [N, 256, 14, 14] antes do dec4
        x = self.dec3(torch.cat([self.up3(x), e3], dim=1))
        x = self.dec2(torch.cat([self.up2(x), e2], dim=1))
        x = self.dec1(torch.cat([self.up1(x), e1], dim=1))
        return self.head(x)


print(UNet()(images).shape)   # [64, 11, 112, 112]

## Treinamento

O laço é o dos classificadores, com a entropia cruzada por pixel. A `evaluate` acumula a matriz de confusão em todo o conjunto de teste.

In [ ]:
def train(model, train_dataloader, num_epochs, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for images, masks in train_dataloader:
            images, masks = images.to(device), masks.to(device)
            loss = F.cross_entropy(model(images), masks)   # logits [N, 11, H, W], alvo [N, H, W]

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        losses.append(total_loss / len(train_dataloader))
        print(f"[{epoch + 1}/{num_epochs}] perda {losses[-1]:.4f}")

    return losses


def evaluate(model, dataloader):
    model.eval()
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
    with torch.no_grad():
        for images, masks in dataloader:
            predictions = model(images.to(device)).argmax(dim=1).cpu()   # [N, H, W]
            confusion += pixel_confusion(predictions, masks)
    return iou_per_class(confusion)

In [ ]:
torch.manual_seed(42)
model = UNet().to(device)
losses = train(model, train_dataloader, num_epochs=8)

plt.plot(range(1, len(losses) + 1), losses)
plt.title("perda de treino da U-Net")
plt.xlabel("época")
plt.ylabel("entropia cruzada")
plt.show()

In [ ]:
iou = evaluate(model, test_dataloader)
print(f"mIoU: {iou.mean():.3f}")
pd.DataFrame({"IoU": iou.numpy()}, index=["fundo"] + [str(d) for d in range(10)]).round(3).T

Desenhando a imagem, a máscara verdadeira e a prevista para algumas imagens de teste:

In [ ]:
test_images, test_masks = next(iter(test_dataloader))
model.eval()
with torch.no_grad():
    predictions = model(test_images[:4].to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(3, 4, figsize=(14, 10.5))
for k in range(4):
    axes[0, k].imshow(test_images[k, 0], cmap="gray")
    axes[0, k].set_title("imagem")
    axes[0, k].axis("off")
    show_mask(axes[1, k], test_masks[k])
    axes[1, k].set_title("máscara real")
    show_mask(axes[2, k], predictions[k])
    axes[2, k].set_title("máscara prevista")
plt.show()

As máscaras previstas quase coincidem com as reais, inclusive nas bordas dos traços. Os erros que sobram são de outro tipo: a forma sai certa, mas o dígito, ou parte dele, recebe a cor errada quando se parece com outro, como um 4 com um 9. É um erro de classificação, e não de segmentação.

## Exercícios

### Exercício 1: sem os atalhos

Escreva uma versão da `UNet` sem as conexões de atalho: o decoder recebe só a saída da convolução transposta, e cada bloco do decoder recebe metade dos canais. Treine com as mesmas épocas e compare a mIoU e as máscaras previstas. O que acontece com os dígitos pequenos?

### Exercício 2: pesos nas classes

Na entropia cruzada, cada pixel de fundo pesa o mesmo que um pixel de dígito, e eles são a grande maioria. Use o argumento `weight` do `F.cross_entropy` para dar ao fundo um peso menor, por exemplo o inverso da frequência de cada classe, e compare a IoU de cada classe.

### Exercício 3: upsampling sem aprender

Troque cada `nn.ConvTranspose2d` por um `nn.Upsample(scale_factor=2, mode="bilinear")` seguido de uma convolução $1 \times 1$ que ajuste os canais. Compare a mIoU e o número de parâmetros das duas versões.